In [1]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose
import contextily as ctx
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from sklearn.cluster import KMeans

import glob
import os
import csv
import ast
import matplotlib.patheffects as path_effects
from matplotlib.ticker import PercentFormatter

### Read in EA Names (Study Area) 

In [2]:
ea_names_select = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/ea_names_select_eas.csv').drop(columns = ['Unnamed: 0'])
ea_names_select = ea_names_select[['ea_code9ch', 'LOC_NAME', 'LOC_NAME_e', 'BASE_NAM']]

### Global EA Names 

In [3]:
global_ea_names = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/global_cesi_names.csv').drop(columns = ['Unnamed: 0'])

### Reading in geojson files 

In [4]:
new_ea = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_ea.json')

new_data_study_area = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_data_study_area.json')

new_sites = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_sites.json')

filt_all_eas_dem_df = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/filt_all_eas_dem_df.json')

## EAs & Grid Metrics  

In [5]:
ea_copy_pqr = new_ea.copy()

ea_copy_pqr = ea_copy_pqr[['ea_code9ch', 'geometry']]

merged_sites_pqr = gpd.sjoin(new_sites, ea_copy_pqr, how = 'inner', predicate = 'intersects')
merged_sites_pqr = merged_sites_pqr[['space_grouping', 'geometry', 'ea_code9ch']].reset_index(drop=True)

In [6]:
### EAs to sites mapping 

In [7]:
grouped_sites_df = merged_sites_pqr.groupby('ea_code9ch').agg({
    'space_grouping': lambda x: list(set(x)),  
})

grouped_sites_df = grouped_sites_df.rename(columns = {'space_grouping':'site_id'})

### Districts from EAs (not using this one) 

In [8]:
district_gdf = new_ea.dissolve(by='DIST_CODE')

district_gdf = district_gdf.reset_index()

### 2021 Census Districts 

In [9]:
dist_2021 = gpd.read_file('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/district_2021_census.geojson')

In [10]:
### Study Area Districts 

In [11]:
dist_2021 = gpd.sjoin(
    dist_2021, merged_sites_pqr.to_crs('EPSG:4326'), 
    how="inner", 
    predicate='contains' 
).reset_index(drop=True).drop(columns=['index_right'])

dist_2021 = dist_2021.drop_duplicates(['Name']).reset_index(drop=True)

### reproject to local CRS 
dist_2021_transformed = dist_2021.to_crs(merged_sites_pqr.crs)

## Outages 

In [12]:
geo_pqr_22 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2022/merged_outage_n_voltage_hourly_22_NEW.csv')
geo_pqr_22 = geo_pqr_22.drop(columns = ['Unnamed: 0'])
geo_pqr_22 = geo_pqr_22[['time', 'site_id', 'outage_events', 'outage_mins']]

In [13]:
geo_pqr_23 = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/Geospatial_Files/2023/merged_outage_n_voltage_hourly_23_NEW.csv')
geo_pqr_23 = geo_pqr_23.drop(columns = ['Unnamed: 0'])
geo_pqr_23 = geo_pqr_23[['time', 'site_id', 'outage_events', 'outage_mins']]

In [14]:
geo_pqr_all = pd.concat([geo_pqr_22, geo_pqr_23], ignore_index=True)
geo_pqr_all['time'] = pd.to_datetime(geo_pqr_all['time'])
geo_pqr_all = geo_pqr_all[~(geo_pqr_all['site_id'] == 0)]

## Remove sites with > `10%` missing data & less than 24 months 

In [15]:
sites_to_omit = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/complete_site_removal_df.csv')['site_id'].to_list()

In [16]:
geo_pqr_all = geo_pqr_all[ ~( geo_pqr_all['site_id'].isin(sites_to_omit) ) ].reset_index(drop=True)

### 1 to 4 hours 

In [17]:
dur_min = 60          # 1 hr
upper_min = 240       # 4 hr

# Keep only outages ≥60 and ≤240 minutes
mask_1to4 = (geo_pqr_all['outage_mins'] >= dur_min) & (geo_pqr_all['outage_mins'] <= upper_min)

# Number of 1-hour blocks
# For 60–240 min, this should be floor(outage_mins / 60)
geo_pqr_all['outage_mins_dur_1to4'] = np.where(
    mask_1to4,
    np.floor(geo_pqr_all['outage_mins'] / 60).astype(int),
    0
)

# Adjust outage events: keep original event count only for 1–4h outages
geo_pqr_all['outage_events_adj_1to4'] = np.where(
    mask_1to4,
    geo_pqr_all['outage_events'],
    0
)

# Group by site
geo_dur_grouped_1to4 = (
    geo_pqr_all
    .groupby('site_id')[['outage_events_adj_1to4', 'outage_mins_dur_1to4']]
    .sum()
    .reset_index()
)

# Rename
geo_dur_grouped_1to4 = geo_dur_grouped_1to4.rename(
    columns={'outage_events_adj_1to4': '1to4hr_outage_events'}
)

# Copy
geo_1to4hr_grouped = geo_dur_grouped_1to4.copy()

In [18]:
geo_1to4hr_grouped.head()

,site_id,1to4hr_outage_events,outage_mins_dur_1to4
0,1.0,31.0,55
1,2.0,42.0,73
2,3.0,464.0,730
3,4.0,413.0,627
4,5.0,30.0,52


### 4+ hour 

In [19]:
dur = 240

geo_pqr_all['outage_mins_dur'] = (
    np.floor(geo_pqr_all['outage_mins'] / dur)
).where(geo_pqr_all['outage_mins'] >= dur, 0)


# Set outage_events to 0 where outage_hours_dur is 0
geo_pqr_all['outage_events_adj'] = geo_pqr_all['outage_events'].where(
    geo_pqr_all['outage_mins_dur'] > 0, 0
)

geo_dur_grouped = geo_pqr_all.groupby('site_id')[['outage_events_adj', 'outage_mins_dur']].sum().reset_index()

## 240min / 4hr 
geo_dur_grouped = geo_dur_grouped.rename(columns = {'outage_events_adj':'240min_outage_events'})

##
geo_4hr_grouped = geo_dur_grouped.copy()

In [20]:
geo_4hr_grouped.head()

,site_id,240min_outage_events,outage_mins_dur
0,1.0,19.0,31.0
1,2.0,21.0,40.0
2,3.0,529.0,1092.0
3,4.0,410.0,811.0
4,5.0,12.0,23.0


### 8+ hour 

In [21]:
dur = 480

geo_pqr_all['outage_mins_dur'] = (
    np.floor(geo_pqr_all['outage_mins'] / dur)
).where(geo_pqr_all['outage_mins'] >= dur, 0)


# Set outage_events to 0 where outage_hours_dur is 0
geo_pqr_all['outage_events_adj'] = geo_pqr_all['outage_events'].where(
    geo_pqr_all['outage_mins_dur'] > 0, 0
)

geo_dur_grouped = geo_pqr_all.groupby('site_id')[['outage_events_adj', 'outage_mins_dur']].sum().reset_index()

## 240min / 4hr 
geo_dur_grouped = geo_dur_grouped.rename(columns = {'outage_events_adj':'480min_outage_events'})

##
geo_8hr_grouped = geo_dur_grouped.copy()

In [22]:
geo_8hr_grouped.head()

,site_id,480min_outage_events,outage_mins_dur
0,1.0,6.0,8.0
1,2.0,13.0,15.0
2,3.0,302.0,382.0
3,4.0,242.0,287.0
4,5.0,4.0,6.0


## Merging: for EAs with multiple sites --> find the average 

In [23]:
def compute_avg_values(site_list, df, event_col):
    geo_lookup = df.set_index('site_id')
    
    # Only use site_ids that are in geo_lookup
    valid_sites = [site for site in site_list if site in geo_lookup.index]
    
    if not valid_sites:
        return pd.Series({
            event_col: np.nan,
            # duration_col: np.nan
        })

    matching = geo_lookup.loc[valid_sites]
    avg_events = matching[event_col].mean()
    
    return pd.Series({
        event_col: avg_events
    })

In [24]:
grouped_sites_df_rez = grouped_sites_df.copy()

In [25]:
##### Outages ##### 

In [26]:
#### 1 to 4 hour outages 

averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_1to4hr_grouped,
        event_col = '1to4hr_outage_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)


#### 4+ hour outages 

averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_4hr_grouped,
        event_col = '240min_outage_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)


#### 8+ hour outages 

averages_df = grouped_sites_df['site_id'].apply(
    lambda site_list: compute_avg_values(
        site_list,
        df = geo_8hr_grouped,
        event_col = '480min_outage_events',
    )
)
grouped_sites_df_rez = grouped_sites_df_rez.join(averages_df)

#### Drop EA with site 83 (NaN) 

In [27]:
grouped_sites_df_rez = grouped_sites_df_rez.dropna()

### Final Merged Results (All CVI & Grid Metrics per EA)  

In [28]:
if new_data_study_area.index.name != 'ea_code9ch':
    new_data_study_area = new_data_study_area.set_index('ea_code9ch')

new_data_study_area_copy = new_data_study_area.copy()[['climate_vul_add', 'climate_vul_geo']]
new_data_study_area_copy = new_data_study_area_copy.join(grouped_sites_df_rez)
new_data_study_area_copy = new_data_study_area_copy.dropna()

In [29]:
new_data_study_area_copy

,climate_vul_add,climate_vul_geo,site_id,1to4hr_outage_events,240min_outage_events,480min_outage_events
ea_code9ch,,,,,,
30100016,0.158602,0.145541,[61],86.0,47.0,14.0
30100018,0.530089,0.515870,[157],39.0,28.0,13.0
30100057,0.483160,0.469408,[114],11.0,4.0,3.0
30100059,0.381087,0.373719,[74],38.0,22.0,6.0
30100061,0.330879,0.327158,[33],113.0,44.0,20.0
...,...,...,...,...,...,...
31300291,0.410119,0.395248,[222],60.0,44.0,15.0
31300310,0.269571,0.256631,[117],53.0,18.0,7.0
31300328,0.241072,0.232259,[296],77.0,25.0,2.0


### CESI Clustering - K_Means 

In [30]:
cesi_df = new_data_study_area_copy.copy()[['climate_vul_add']]
cesi_values = cesi_df[['climate_vul_add']].values


# Run KMeans with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42, n_init='auto')
cesi_df['cesi_cluster'] = kmeans.fit_predict(cesi_values)


# Map clusters to 'Low', 'High' based on cluster centers
# Sort cluster labels by increasing vulnerability
cluster_order = kmeans.cluster_centers_.flatten().argsort()
label_map = {cluster: label for cluster, label in zip(cluster_order, ['Low', 'High'])}
cesi_df['cesi_level'] = cesi_df['cesi_cluster'].map(label_map)
cesi_df = cesi_df.drop(columns=['cesi_cluster'])

In [31]:
cesi_df.head()

,climate_vul_add,cesi_level
ea_code9ch,,
30100016,0.158602,Low
30100018,0.530089,High
30100057,0.483160,High
30100059,0.381087,Low
30100061,0.330879,Low


In [32]:
cesi_df['cesi_level'].value_counts()

cesi_level
Low     239
High     85
Name: count, dtype: int64

In [33]:
cesi_df.groupby('cesi_level')['climate_vul_add'].describe()

,count,mean,std,min,25%,50%,75%,max
cesi_level,,,,,,,,
High,85.0,0.510824,0.079525,0.410119,0.449924,0.505242,0.548010,0.775336
Low,239.0,0.307465,0.054696,0.158602,0.267160,0.304269,0.353617,0.407388


In [34]:
# Low (0.158 - 0.407); n = 239 
# High (0.410 - 0.775); n = 85 

### Assign output of K_Means Clustering back to df 

In [35]:
new_data_study_area_copy = new_data_study_area_copy.join(cesi_df.drop(columns = ['climate_vul_add']))

### Only keep the 214 EAs in the TPLW dataset (for consistency in analysis) 

In [36]:
tplw_ea_df = pd.read_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/unique_ea_codes_TPLW.csv')

tplw_ea_list = tplw_ea_df['ea_code9ch'].tolist()

In [37]:
new_data_study_area_copy = new_data_study_area_copy[new_data_study_area_copy.index.isin(tplw_ea_list)]

In [38]:
new_data_study_area_copy.head()

,climate_vul_add,climate_vul_geo,site_id,1to4hr_outage_events,240min_outage_events,480min_outage_events,cesi_level
ea_code9ch,,,,,,,
30200002,0.289649,0.280136,[456],44.000000,25.000000,9.0,Low
30200012,0.285566,0.280706,[430],59.000000,61.000000,32.0,Low
30200013,0.268086,0.253484,[363],73.000000,50.000000,25.0,Low
30200014,0.235367,0.209017,[301],27.000000,8.000000,2.0,Low
30200015,0.236200,0.216458,"[104, 32, 455]",32.666667,15.666667,5.0,Low


In [39]:
new_data_study_area_copy['cesi_level'].value_counts()

cesi_level
Low     159
High     55
Name: count, dtype: int64

In [40]:
# Low (0.191 - 0.407); n = 159 
# High (0.410 - 0.709); n = 55 

In [41]:
new_data_study_area_copy.groupby('cesi_level')['climate_vul_add'].describe()

,count,mean,std,min,25%,50%,75%,max
cesi_level,,,,,,,,
High,55.0,0.508873,0.072887,0.410119,0.451484,0.500327,0.551242,0.708665
Low,159.0,0.303900,0.053968,0.191265,0.260545,0.297413,0.343944,0.407388


### Save to csv - `rev_1` 

In [42]:
# new_data_study_area_copy[['cesi_level']].reset_index()

In [43]:
# new_data_study_area_copy[['cesi_level']].reset_index().to_csv("/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/eas_cesi_level_214_eas_rev1.csv")

### Save full csv to file 

In [48]:
new_data_study_area_trimmed = new_data_study_area_copy.reset_index().drop(columns = ['climate_vul_geo', '1to4hr_outage_events', '1to4hr_outage_events', '240min_outage_events', '480min_outage_events'])

In [51]:
# new_data_study_area_trimmed.to_csv('/work/pi_jtaneja_umass_edu/kdonkor_umass_edu/__KDee_Project_Git__/Geospatial_Analysis/Hotspot Analysis_Bivariate_Maps_Box_plots/new_data_study_area_trimmed.csv')